In [1]:
from torch.utils.data import DataLoader, random_split
from utils.report import AverageMeter
from utils.metrics import calculate_accuracy

import os
import random
import torch
import warnings

import numpy as np
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data


warnings.filterwarnings('ignore')

In [2]:
BATCH_SIZE = 256
LR = 0.001
ES_THRES = 5
EPOCHS = 12
SEED = 1234
GAMMA = 10
OUTPUT_DIR = "./state_dicts/DANN_SVHN_MNIST"

# Set Seed

In [3]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)

# Download MNIST, SVHN

In [4]:
from torchvision.datasets import MNIST, SVHN
import torchvision.transforms as transforms

In [5]:
download_root_mnist = '../datasets/MNIST_DATASET'
train_data_mnist = MNIST(download_root_mnist, train=True, download=True)
mean_mnist = train_data_mnist.data.float().mean() / 255
std_mnist = train_data_mnist.data.float().std() / 255

print(f'Calculated mean: {mean_mnist}')
print(f'Calculated std: {std_mnist}')

Calculated mean: 0.13066047430038452
Calculated std: 0.30810779333114624


In [6]:
train_transforms_mnist = transforms.Compose([
                            transforms.RandomRotation(5, fill=(0,)),
                            transforms.RandomCrop(28, padding = 2),
                            transforms.Grayscale(num_output_channels=3),
                            transforms.Resize(32),
                            transforms.ToTensor(),
                            transforms.Normalize(mean = [mean_mnist], std = [std_mnist])])

test_transforms_mnist = transforms.Compose([
                            transforms.Grayscale(num_output_channels=3),
                            transforms.Resize(32),
                            transforms.ToTensor(),
                            transforms.Normalize(mean = [mean_mnist], std = [std_mnist])])

In [7]:
train_valid_dataset_mnist = MNIST(download_root_mnist, transform=train_transforms_mnist, train=True, download=True)
train_dataset_mnist, valid_dataset_mnist = random_split(train_valid_dataset_mnist, [54000, 6000])
test_dataset_mnist = MNIST(download_root_mnist, transform=test_transforms_mnist, train=False, download=True)

In [8]:
download_root_svhn = '../datasets/SVHN_DATASET'
train_data_svhn = SVHN(download_root_svhn, split="train", download=True)
mean_svhn = train_data_svhn.data.mean() / 255
std_svhn = train_data_svhn.data.std() / 255

print(f'Calculated mean: {mean_svhn}')
print(f'Calculated std: {std_svhn}')

Using downloaded and verified file: ../datasets/SVHN_DATASET/train_32x32.mat
Calculated mean: 0.45141874380092256
Calculated std: 0.19929124669110937


In [9]:
train_transforms_svhn = transforms.Compose([
                            transforms.ToTensor(),
                            transforms.Normalize(mean = [mean_svhn], std = [std_svhn])])

test_transforms_svhn = transforms.Compose([
                            transforms.ToTensor(),
                            transforms.Normalize(mean = [mean_svhn], std = [std_svhn])])

In [10]:
train_valid_dataset_svhn = SVHN(download_root_svhn, transform=train_transforms_svhn, split="train", download=True)
train_dataset_svhn, valid_dataset_svhn = random_split(train_valid_dataset_svhn, [65931, 7326])
test_dataset_svhn = SVHN(download_root_svhn, transform=test_transforms_svhn, split="test", download=True)

Using downloaded and verified file: ../datasets/SVHN_DATASET/train_32x32.mat
Using downloaded and verified file: ../datasets/SVHN_DATASET/test_32x32.mat


# Model

In [11]:
class FeatureExtractor(nn.Module):
    def __init__(self, in_channel_dim=3, out_channel_dim=16):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel_dim, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=out_channel_dim, kernel_size=5)
        
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
#         x = x.view(-1, 16 * 5 * 5)

        return x

In [12]:
class Classifier(nn.Module):
    def __init__(self, in_dim, out_dim=10):
        super(Classifier, self).__init__()
        self.fc1 = nn.Linear(in_dim, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, out_dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        
        return x

In [13]:
class Discriminator(nn.Module):
    def __init__(self, in_dim, out_dim=1):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear(in_dim, 84)
        self.fc2 = nn.Linear(84, out_dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.sigmoid(x)
        
        return x

In [14]:
class GradientReversalLayer(autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x
    
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

# Train/Eval Functions

In [15]:
def train_loop_classifier(feature_extractor, classifier, discriminator, grl, 
                          source_loader, target_loader, loss_func_classifier, loss_func_discriminator, optimizer_classifier, optimizer_discriminator,
                          epoch, summary_loss_classifier, summary_loss_discriminator_source, summary_loss_discriminator_target, summary_acc_classifier=None, device=None):
    feature_extractor.train()
    classifier.train()
    discriminator.train()
    alpha = 0
    for batch_idx, ((data, label), (data_target, _)) in enumerate(zip(source_loader, target_loader)):
        if device is not None:
            data, label, data_target = data.to(device), label.to(device), data_target.to(device)
            
        optimizer_classifier.zero_grad()
        optimizer_discriminator.zero_grad()
        
        p = float(batch_idx + epoch * len(source_loader)) / (EPOCHS * len(source_loader))
#         p = float(epoch) / (EPOCHS)
        alpha = (2. / (1. + np.exp(-GAMMA * p)) - 1)
        
        # Train classifier with source data
        feature = feature_extractor(data)
        feature = torch.flatten(feature, 1)
        output_classifier = classifier(feature)
        loss_classifier = loss_func_classifier(output_classifier, label)
        loss_classifier.backward()
        optimizer_classifier.step()
        
        # Train discriminator with source data
        feature = feature_extractor(data)
        feature = torch.flatten(feature, 1)
        output_discriminator = discriminator(grl(feature, alpha))
        loss_discriminator = loss_func_discriminator(output_discriminator, torch.zeros(output_discriminator.shape).to(device))
        
        # Train discriminator with target data
        feature_target = feature_extractor(data_target)
        feature_target = torch.flatten(feature_target, 1)
        output_discriminator_target = discriminator(grl(feature_target, alpha))
        loss_discriminator_target = loss_func_discriminator(output_discriminator_target, torch.ones(output_discriminator_target.shape).to(device))
        
        loss_discriminator = loss_discriminator + loss_discriminator_target
        loss_discriminator.backward()
        optimizer_discriminator.step()
        
        summary_loss_classifier.update(loss_classifier.detach().item(), BATCH_SIZE)
        summary_loss_discriminator_source.update(loss_discriminator.detach().item(), BATCH_SIZE)
        summary_loss_discriminator_target.update(loss_discriminator_target.detach().item(), BATCH_SIZE)
        if summary_acc_classifier is not None:
            acc = calculate_accuracy(output_classifier, label)
            summary_acc_classifier.update(acc, BATCH_SIZE)
    print(alpha)
            
            
def eval_loop_classifier(feature_extractor, classifier, discriminator,
                         source_loader, target_loader, loss_func_classifier, loss_func_discriminator,
                         summary_loss_classifier, summary_loss_classifier_target, summary_acc_classifier=None, summary_acc_classifier_target=None, device=None):
    feature_extractor.eval()
    classifier.eval()
    discriminator.eval()
    
    with torch.no_grad():
        for batch_idx, ((data, label), (data_target, label_target)) in enumerate(zip(source_loader, target_loader)):
            if device is not None:
                data, label, data_target, label_target = data.to(device), label.to(device), data_target.to(device), label_target.to(device)

            # Source classifiying results
            feature = feature_extractor(data)
            feature = torch.flatten(feature, 1)
            output_classifier = classifier(feature)
            loss_classifier = loss_func_classifier(output_classifier, label)
            
            # Target classifying results
            feature = feature_extractor(data_target)
            feature = torch.flatten(feature, 1)
            output_classifier_target = classifier(feature)
            loss_classifier_target = loss_func_classifier(output_classifier_target, label_target)
            
            feature_discriminator = feature_extractor(torch.cat([data, data_target], dim=0))
            feature_discriminator = torch.flatten(feature_discriminator, 1)
            
            output_discriminator = discriminator(feature_discriminator)
            loss_discriminator = loss_func_discriminator(output_discriminator, torch.cat([torch.zeros((data.shape[0], 1)).to(device), torch.ones((data_target.shape[0], 1)).to(device)], dim=0))
            
            summary_loss_classifier.update(loss_classifier.detach().item(), BATCH_SIZE)
            summary_loss_classifier_target.update(loss_classifier_target.detach().item(), BATCH_SIZE)
            if summary_acc_classifier is not None:
                acc = calculate_accuracy(output_classifier, label)
                summary_acc_classifier.update(acc, BATCH_SIZE)
                
                acc = calculate_accuracy(output_classifier_target, label_target)
                summary_acc_classifier_target.update(acc, BATCH_SIZE)


# Training with Early Stopping

In [16]:
def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    feature_extractor = FeatureExtractor()
    classifier = Classifier(400)
    discriminator = Discriminator(400)
    grl = GradientReversalLayer.apply
    
    feature_extractor.to(device)
    classifier.to(device)
    discriminator.to(device)
    
    criterion_classifier = nn.CrossEntropyLoss()
    criterion_discriminator = nn.BCELoss()
    
    model_parameters_classifier = list(feature_extractor.parameters()) + list(classifier.parameters())
    optimizer_classifier = optim.Adam(model_parameters_classifier, lr=LR)
    model_parameters_discriminator = list(feature_extractor.parameters()) + list(discriminator.parameters())
    optimizer_discriminator = optim.Adam(model_parameters_discriminator, lr=LR)
    
    source_train_loader = DataLoader(dataset=train_dataset_svhn, batch_size=BATCH_SIZE)
    target_train_loader = DataLoader(dataset=train_dataset_mnist, batch_size=BATCH_SIZE)
    
    best_epoch = None
    best_loss = None
    epoch = 0
    es_count = 0
    while(epoch < EPOCHS):
        epoch += 1
        summary_loss_classifier_train = AverageMeter()
        summary_acc_classifier_train = AverageMeter()
        summary_loss_discriminator_source_train = AverageMeter()
        summary_loss_discriminator_target_train = AverageMeter()
        
        train_loop_classifier(feature_extractor, classifier, discriminator, grl, 
                              source_train_loader, target_train_loader, criterion_classifier, criterion_discriminator, optimizer_classifier, optimizer_discriminator,
                              epoch-1, summary_loss_classifier_train, summary_loss_discriminator_source_train, summary_loss_discriminator_target_train, summary_acc_classifier_train, device=device)

        source_valid_loader = DataLoader(dataset=valid_dataset_svhn, batch_size=BATCH_SIZE)
        target_valid_loader = DataLoader(dataset=valid_dataset_mnist, batch_size=BATCH_SIZE)

        summary_loss_classifier_valid = AverageMeter()
        summary_loss_classifier_target_valid = AverageMeter()
        summary_acc_classifier_valid = AverageMeter()
        summary_acc_classifier_target_valid = AverageMeter()

        eval_loop_classifier(feature_extractor, classifier, discriminator,
                             source_valid_loader, target_valid_loader, criterion_classifier, criterion_discriminator,
                             summary_loss_classifier_valid, summary_loss_classifier_target_valid, summary_acc_classifier_valid, summary_acc_classifier_target_valid, device=device)

        print(f"#### EPOCH {epoch}/{EPOCHS} ####")
        print(f"[train loss]{summary_loss_classifier_train.avg:.3f} [train acc]{summary_acc_classifier_train.avg:.3f} [train discrmt. source loss] {summary_loss_discriminator_source_train.avg:.3f} [train discrmt. target loss]{summary_loss_discriminator_target_train.avg:.3f}")
        print(f"[valid source loss]{summary_loss_classifier_valid.avg:.3f} [valid target loss]{summary_loss_classifier_target_valid.avg:.3f} [valid source acc]{summary_acc_classifier_valid.avg:.3f} [valid target acc]{summary_acc_classifier_target_valid.avg:.3f}")
        print(f"#######################")
        if best_loss is None:
            best_loss = summary_acc_classifier_valid.avg
            best_epoch = epoch

        if best_loss > summary_acc_classifier_valid.avg:
            best_loss = summary_acc_classifier_valid.avg
            best_epoch = epoch
            es_count = 0
        else:
            es_count += 1

    print(f"Best epoch: {best_epoch}")
    return best_epoch

In [17]:
best_epoch = run_training()

KeyboardInterrupt: 

In [18]:
def run_second_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    feature_extractor = FeatureExtractor()
    classifier = Classifier(400)
    discriminator = Discriminator(400)
    grl = GradientReversalLayer.apply
    
    feature_extractor.to(device)
    classifier.to(device)
    discriminator.to(device)
    
    criterion_classifier = nn.CrossEntropyLoss()
    criterion_discriminator = nn.BCELoss()
    
    model_parameters_classifier = list(feature_extractor.parameters()) + list(classifier.parameters())
    optimizer_classifier = optim.Adam(model_parameters_classifier, lr=LR)
    model_parameters_discriminator = list(feature_extractor.parameters()) + list(discriminator.parameters())
    optimizer_discriminator = optim.Adam(model_parameters_discriminator, lr=LR)
    
    source_train_loader = DataLoader(dataset=train_valid_dataset_svhn, batch_size=BATCH_SIZE)
    target_train_loader = DataLoader(dataset=train_valid_dataset_mnist, batch_size=BATCH_SIZE)
    
    best_epoch = None
    best_loss = None
    epoch = 0
    es_count = 0
    while(epoch < EPOCHS):
        epoch += 1
        summary_loss_classifier_train = AverageMeter()
        summary_acc_classifier_train = AverageMeter()
        summary_loss_discriminator_source_train = AverageMeter()
        summary_loss_discriminator_target_train = AverageMeter()

        train_loop_classifier(feature_extractor, classifier, discriminator, grl, 
                              source_train_loader, target_train_loader, criterion_classifier, criterion_discriminator, optimizer_classifier, optimizer_discriminator,
                              epoch-1, summary_loss_classifier_train, summary_loss_discriminator_source_train, summary_loss_discriminator_target_train, summary_acc_classifier_train, device=device)
        
        print(f"#### EPOCH {epoch} ####")
        print(f"[train loss]{summary_loss_classifier_train.avg:.3f} [train acc]{summary_acc_classifier_train.avg:.3f} [train discrmt. source loss] {summary_loss_discriminator_source_train.avg:.3f} [train discrmt. target loss]{summary_loss_discriminator_target_train.avg:.3f}")
    return feature_extractor, classifier

In [19]:
feature_extractor, classifier = run_second_training()

0.3272285195379625
#### EPOCH 1 ####
EPOCH:1, [train loss]1.475 [train acc]0.498 [train discrmt. source loss] 0.711 [train discrmt. target loss]0.380
0.638944444408946
#### EPOCH 2 ####
EPOCH:2, [train loss]0.675 [train acc]0.803 [train discrmt. source loss] 0.366 [train discrmt. target loss]0.141
0.8252489336346909
#### EPOCH 3 ####
EPOCH:3, [train loss]0.602 [train acc]0.824 [train discrmt. source loss] 0.443 [train discrmt. target loss]0.165
0.9201065823971515
#### EPOCH 4 ####
EPOCH:4, [train loss]0.524 [train acc]0.847 [train discrmt. source loss] 0.336 [train discrmt. target loss]0.124
0.9644761236734174
#### EPOCH 5 ####
EPOCH:5, [train loss]0.450 [train acc]0.868 [train discrmt. source loss] 0.215 [train discrmt. target loss]0.079
0.9844047698370935
#### EPOCH 6 ####
EPOCH:6, [train loss]0.411 [train acc]0.880 [train discrmt. source loss] 0.150 [train discrmt. target loss]0.057
0.9931923273411905
#### EPOCH 7 ####
EPOCH:7, [train loss]0.379 [train acc]0.888 [train discrmt. sour

In [20]:
def run_test(feature_extractor, classifier):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    feature_extractor.eval()
    classifier.eval()
    
    test_loader = DataLoader(dataset=test_dataset_mnist, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    
    summary_loss_test = AverageMeter()
    summary_acc_test = AverageMeter()

    summary_loss_test, summary_acc_test = eval_loop_classifier(feature_extractor, classifier, test_loader, 
                                               criterion, None, summary_loss_test, summary_acc_test, device=device)

    print(f"[test loss]{summary_loss_test.avg} [test acc]{summary_acc_test.avg}")

In [21]:
run_test(feature_extractor, classifier)

TypeError: eval_loop_classifier() missing 2 required positional arguments: 'summary_loss_classifier' and 'summary_loss_classifier_target'